# 🔬 Deep Research Agent - Multi-Agent AI Research System

This notebook demonstrates building a production-ready AI research agent that:
- Plans research strategies automatically
- Executes parallel web searches
- Synthesizes findings into comprehensive reports
- Delivers results via email

**Tech Stack:** OpenAI Agents SDK | Python asyncio | Pydantic | SendGrid

---
## Step 1: Import Required Libraries

In [1]:
# OpenAI Agents SDK imports for multi-agent orchestration
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings

# Pydantic for structured outputs (ensures consistent data formats)
from pydantic import BaseModel, Field

# Standard Python libraries
from dotenv import load_dotenv  # Load environment variables
import asyncio  # For parallel processing
import os
from typing import Dict

# SendGrid for email functionality
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

# Jupyter display utilities
from IPython.display import display, Markdown

---
## Step 2: Load Environment Variables

Required in `.env` file:
- `OPENAI_API_KEY`
- `SENDGRID_API_KEY`

In [2]:
load_dotenv(override=True)

True

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

---
## Step 3: Create Search Agent (Agent #1)

This agent searches the web and returns concise summaries (2-3 paragraphs, <300 words)

In [3]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],  # WebSearchTool for web searching
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),  # Force tool usage
)

### Test the Search Agent

Run a simple search to see how it works

In [ ]:
message = "Latest AI Agent frameworks in 2025"

# trace() logs execution to OpenAI platform for debugging
with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

---
## Step 4: Create Planner Agent (Agent #2) with Structured Outputs

This agent plans which searches to perform for a given query

In [4]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 2

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

# Define structure for a single search item
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


# Define structure for the complete search plan
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


# Planner agent returns structured WebSearchPlan output
planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,  # Ensures structured output
)

### Test the Planner Agent

See what search queries it generates

In [ ]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

---
## Step 5: Create Email Tool

Custom function that agents can call to send emails via SendGrid

In [5]:
# @function_tool decorator registers this as a tool agents can use
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("n8n3automation@gmail.com") # Change this to your verified email
    to_email = To("kvishnudharshan@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

### Verify the Email Tool Registration

In [6]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7c5655e9bec0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

---
## Step 6: Create Email Agent (Agent #3)

This agent converts markdown reports to HTML and sends them via email

In [7]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],  # Give access to the email sending tool
    model="gpt-4o-mini",
)


---
## Step 7: Create Writer Agent (Agent #4) with Structured Output

This agent synthesizes search results into a comprehensive report (1000+ words)

In [8]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


# Define structured output format for the report
class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,  # Ensures structured report output
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

---
## Step 8: Define Workflow Functions - Search Planning & Execution

These functions orchestrate the search process

In [9]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    # Create tasks for parallel execution (all searches run simultaneously!)
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    # Wait for all searches to complete
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

---
## Step 9: Define Workflow Functions - Report Writing & Email Delivery

In [10]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email_report(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

---
## Step 10: Run the Complete Multi-Agent Research Pipeline! 🚀

This orchestrates all 4 agents:
1. **Planner Agent** → Decides which searches to perform
2. **Search Agents** → Execute searches in parallel
3. **Writer Agent** → Synthesizes findings into a report
4. **Email Agent** → Delivers the report

**Watch the console output to see each step execute!**

In [11]:
query ="Latest AI Agent frameworks in 2025"

# trace() allows you to view execution at https://platform.openai.com/traces
with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email_report(report)  
    print("Hooray!")




Starting research...
Planning searches...
Will perform 2 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


### As always, take a look at the trace

https://platform.openai.com/traces

---
---
# 🎉 Congratulations!

You've built a production-ready multi-agent research system!

## What You've Learned:
✅ **Multi-agent orchestration** - Coordinating 4 specialized agents  
✅ **Parallel processing** - Running tasks simultaneously with asyncio  
✅ **Structured outputs** - Using Pydantic for reliable data formats  
✅ **Custom tools** - Creating email sending functionality  
✅ **Web search integration** - Using OpenAI's hosted search tool  
✅ **Tracing & debugging** - Monitoring agent execution on OpenAI platform  

## The Complete Agent Workflow:
```
User Query
    ↓
Planner Agent (decides what to search)
    ↓
Search Agents (parallel web searches)
    ↓
Writer Agent (synthesizes into report)
    ↓
Email Agent (delivers via email)
    ↓
Done! ✅
```

## Real-World Applications:
- 📊 Market research automation
- 💼 Investment due diligence
- 📰 Content research for writers
- 🎓 Academic research assistance
- 📈 Competitive intelligence

---
**Created by:** VishnuDharshan K  
**LinkedIn:** [linkedin.com/in/vishnu-dharshan-k](https://linkedin.com/in/vishnu-dharshan-k)  
**GitHub:** [Your GitHub Link]

⭐ Star this repo if you found it helpful!